In [1]:
import statsmodels
import torch
import torch.nn as nn
import torch.optim as optim
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
import numpy as np

In [2]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Global seed set to {SEED}")

NameError: name 'np' is not defined

In [ ]:
data = pd.read_csv("../data/processed_vol_features.csv", index_col=0, parse_dates=True)

In [ ]:
data.head()

,Close,log_return,realized_vol5,realized_vol10,realized_vol21,vol_of_vol5,parkinson,gk_vol,rv_lag1,rv_lag5,rv_lag21,vix,vix_change,mom_5d,mom_21d,target_rv
Date,,,,,,,,,,,,,,,,
2020-02-11,306.383240,0.001732,0.100497,0.149352,0.133205,0.810295,0.078104,0.010284,0.125554,0.197924,0.072707,15.18,0.009309,0.018842,0.029320,0.080312
2020-02-12,308.357208,0.006422,0.080312,0.149226,0.132990,0.774494,0.076915,0.009709,0.100497,0.208508,0.080210,13.74,-0.094862,0.013700,0.028876,0.084153
2020-02-13,308.028229,-0.001067,0.084153,0.150795,0.132874,0.345296,0.077899,0.015213,0.080312,0.208490,0.075032,14.15,0.029840,0.009222,0.029348,0.056829
2020-02-14,308.521759,0.001601,0.056829,0.095641,0.132839,0.404818,0.077526,0.010428,0.084153,0.125535,0.071916,13.68,-0.033216,0.016256,0.028672,0.054499
2020-02-18,307.726685,-0.002580,0.054499,0.101056,0.131008,0.308663,0.078824,0.012618,0.056829,0.125554,0.078410,14.83,0.084064,0.006126,0.017557,0.060254


In [ ]:
data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1018 entries, 2020-02-11 to 2025-12-26
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Close           1018 non-null   float64
 1   log_return      1018 non-null   float64
 2   realized_vol5   1018 non-null   float64
 3   realized_vol10  1018 non-null   float64
 4   realized_vol21  1018 non-null   float64
 5   vol_of_vol5     1018 non-null   float64
 6   parkinson       1018 non-null   float64
 7   gk_vol          1018 non-null   float64
 8   rv_lag1         1018 non-null   float64
 9   rv_lag5         1018 non-null   float64
 10  rv_lag21        1018 non-null   float64
 11  vix             1018 non-null   float64
 12  vix_change      1018 non-null   float64
 13  mom_5d          1018 non-null   float64
 14  mom_21d         1018 non-null   float64
 15  target_rv       1018 non-null   float64
dtypes: float64(16)
memory usage: 135.2 KB


In [ ]:
series = data["realized_vol21"].dropna()

split  = int(len(series) * 0.85)   # 85% train, 15% test

train  = series.iloc[:split]
test   = series.iloc[split:]

print(f"Train: {len(train)} obs  "
      f"({train.index[0].date()} → {train.index[-1].date()})")
print(f"Test : {len(test)}  obs  "
      f"({test.index[0].date()} → {test.index[-1].date()})")

Train: 865 obs  (2020-02-11 → 2024-11-26)
Test : 153  obs  (2024-11-27 → 2025-12-26)


In [ ]:
import numpy as np

In [ ]:
p, d, q = 2, 0, 1    

history = list(train)
predictions = []

print(f"Running walk-forward forecast "
      f"ARIMA({p},{d},{q}) on {len(test)} steps...")

for i, actual in enumerate(test):
    # Fit on all available history
    model = ARIMA(history, order=(p, d, q))
    fit   = model.fit()

    # Forecast 1 step ahead
    pred  = fit.forecast(steps=1)[0]
    predictions.append(pred)

    # Add actual value to history (walk-forward)
    history.append(actual)

    if i % 50 == 0:
        print(f"  Step {i+1}/{len(test)}  "
              f"pred={pred:.2f}  actual={actual:.2f}")

predictions = np.array(predictions)
actuals  = test.values

Running walk-forward forecast ARIMA(2,0,1) on 153 steps...


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'


  Step 1/153  pred=0.14  actual=0.14


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

  Step 51/153  pred=0.52  actual=0.52


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

  Step 101/153  pred=0.10  actual=0.10


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

  Step 151/153  pred=0.11  actual=0.10


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
#evaluation
mae  = mean_absolute_error(actuals, predictions)
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mape = np.mean(np.abs((actuals - predictions) /
               (actuals + 1e-8))) * 100

# Directional accuracy — did vol go up/down correctly?
actual_dir = np.sign(np.diff(actuals))
pred_dir   = np.sign(np.diff(predictions))
dir_acc    = np.mean(actual_dir == pred_dir) * 100

print(f"  ARIMA({p},{d},{q}) — Baseline Results")
print(f"  MAE             : {mae:.4f}")
print(f"  RMSE            : {rmse:.4f}")
print(f"  MAPE            : {mape:.2f}%")
print(f"  Directional Acc : {dir_acc:.2f}%")

  ARIMA(2,0,1) — Baseline Results
  MAE             : 0.0113
  RMSE            : 0.0259
  MAPE            : 7.07%
  Directional Acc : 53.95%


In [ ]:
# #NaiveBaseline
# class NaiveBaseline:
#     """Predict last known value (or last N values repeated)"""
#     def predict(self, X):
#         # X shape: (batch, seq_len, features)
#         return X[:, -1, :]  # repeat last timestep

# # Evaluate
# with torch.no_grad():
#     naive = NaiveBaseline()
#     preds = naive.predict(test_X)  # (N, 1)
#     mae = torch.mean(torch.abs(preds - test_y.squeeze(-1)))
#     print(f"Naïve MAE: {mae:.4f}")


# ## Moving Average Baseline
# class MovingAverageBaseline:
#     def __init__(self, window=7):
#         self.window = window

#     def predict(self, X):
#         # X: (batch, seq_len, 1)
#         return X[:, -self.window:, :].mean(dim=1)  # avg of last `window` steps

# ma = MovingAverageBaseline(window=7)
# preds = ma.predict(test_X)

### Deep Learning Forecasting

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from sklearn.preprocessing import MinMaxScaler

In [ ]:
class VolDataset(Dataset):
    def __init__(self, dataframe, feature_cols, target_col, seq_len, horizon = 1, scaler_X = None, scaler_y = None, fit_scalers=False):
        self.df = dataframe
        self.seq_len = seq_len
        self.horizon = horizon
        X_raw = self.df[feature_cols].values.astype(np.float32)
        y_raw = self.df[[target_col]].values.astype(np.float32)
        if fit_scalers:
            self.scaler_X = MinMaxScaler()
            self.scaler_y = MinMaxScaler()
            X_scaled = self.scaler_X.fit_transform(X_raw)
            y_scaled = self.scaler_y.fit_transform(y_raw)
        else:
            self.scaler_X = scaler_X
            self.scaler_y = scaler_y
            X_scaled = self.scaler_X.transform(X_raw)
            y_scaled = self.scaler_y.transform(y_raw)
        self.X, self.y = self._make_sequences(X_scaled, y_scaled)
        
    def _make_sequences(self, X, y):
        X_seq, y_seq = [], []
        total = len(X) - self.seq_len - self.horizon + 1
        for i in range(total):
            X_seq.append(X[i:i+self.seq_len])
            y_seq.append(y[i+self.seq_len+self.horizon-1])
        return (torch.tensor(np.array(X_seq), dtype=torch.float32), torch.tensor(np.array(y_seq), dtype=torch.float32))
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # row = self.df.iloc[idx]
        # data = row.drop("realized_vol21").values.astype(np.float32)
        # target = row["realized_vol21"].astype(np.float32)
        return self.X[idx], self.y[idx]

In [ ]:
feature_cols = [col for col in data.columns if col != "realized_vol21"]
target_col = "realized_vol21"
n = len(data)
train_end = int(n * 0.7)
val_end = int(n * 0.85)
train_data = data.iloc[:train_end]
val_data = data.iloc[train_end:val_end]
test_data = data.iloc[val_end:]

In [ ]:
train_dataset = VolDataset(train_data, feature_cols, target_col, seq_len=30, horizon=1, fit_scalers=True)
val_dataset = VolDataset(val_data, feature_cols, target_col, seq_len=30, horizon=1, 
                         scaler_X=train_dataset.scaler_X, scaler_y=train_dataset.scaler_y)
test_dataset = VolDataset(test_data, feature_cols, target_col, seq_len=30, horizon=1, 
                          scaler_X=train_dataset.scaler_X, scaler_y=train_dataset.scaler_y)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

### MLP Forecaster

In [ ]:
class MLPForecaster(nn.Module):
    def __init__(self, seq_len, num_features, hidden_dims, horizon=1, dropout=0.2):
        super().__init__()
        input_dim = seq_len * num_features
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(in_dim, h),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.BatchNorm1d(h)
            ]
            in_dim = h
        layers += [nn.Linear(in_dim, horizon)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten (batch, seq_len*num_features)
        return self.net(x).squeeze()  # (batch, horizon) -> (batch)


In [ ]:
model_mlp = MLPForecaster(seq_len=30, num_features=data.shape[1]-1, hidden_dims=[128, 64], horizon=1, dropout=0.2)

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3,
                optimizer_name="adam", criterion_name="mse",
                use_scheduler=False, clip_grad=None, model_name="model"):
    """
    Unified training loop.
    
    Args:
        optimizer_name : "adam" | "adamw" | "rmsprop"
        criterion_name : "mse"  | "huber"
        use_scheduler  : ReduceLROnPlateau if True
        clip_grad      : max grad norm (None = disabled)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.HuberLoss() if criterion_name == "huber" else nn.MSELoss()

    if optimizer_name == "adamw":
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    elif optimizer_name == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr)

    scheduler = (optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
                 if use_scheduler else None)

    best_val_loss = float("inf")
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b.squeeze())
            loss.backward()
            if clip_grad:
                nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_v, y_v in val_loader:
                X_v, y_v = X_v.to(device), y_v.to(device)
                val_losses.append(criterion(model(X_v), y_v.squeeze()).item())

        avg_train = np.mean(train_losses)
        avg_val   = np.mean(val_losses)

        if scheduler:
            scheduler.step(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), f"best_{model_name}.pth")

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch+1}/{epochs}  Train: {avg_train:.4f}  Val: {avg_val:.4f}  LR: {current_lr:.2e}")

    model.load_state_dict(torch.load(f"best_{model_name}.pth"))
    return model

In [ ]:

trained_mlp = train_model(model_mlp, train_loader, val_loader, epochs=20, lr=1e-3, model_name="MLPForecaster")

Epoch 1/20  Train: 0.2784  Val: 0.0113  LR: 1.00e-03
Epoch 2/20  Train: 0.1585  Val: 0.0040  LR: 1.00e-03
Epoch 3/20  Train: 0.1069  Val: 0.0137  LR: 1.00e-03
Epoch 4/20  Train: 0.0889  Val: 0.0166  LR: 1.00e-03
Epoch 5/20  Train: 0.0890  Val: 0.0190  LR: 1.00e-03
Epoch 6/20  Train: 0.0734  Val: 0.0157  LR: 1.00e-03
Epoch 7/20  Train: 0.0691  Val: 0.0085  LR: 1.00e-03
Epoch 8/20  Train: 0.0506  Val: 0.0039  LR: 1.00e-03
Epoch 9/20  Train: 0.0533  Val: 0.0065  LR: 1.00e-03
Epoch 10/20  Train: 0.0413  Val: 0.0037  LR: 1.00e-03
Epoch 11/20  Train: 0.0466  Val: 0.0024  LR: 1.00e-03
Epoch 12/20  Train: 0.0425  Val: 0.0025  LR: 1.00e-03
Epoch 13/20  Train: 0.0414  Val: 0.0052  LR: 1.00e-03
Epoch 14/20  Train: 0.0320  Val: 0.0025  LR: 1.00e-03
Epoch 15/20  Train: 0.0238  Val: 0.0017  LR: 1.00e-03
Epoch 16/20  Train: 0.0238  Val: 0.0038  LR: 1.00e-03
Epoch 17/20  Train: 0.0171  Val: 0.0071  LR: 1.00e-03
Epoch 18/20  Train: 0.0200  Val: 0.0064  LR: 1.00e-03
Epoch 19/20  Train: 0.0167  Val: 0.00

In [ ]:
def evaluate_model(model, test_loader, scaler_y, model_name="Model"):
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        preds, actuals = [], []
        for X_test, y_test in test_loader:
            X_test, y_test = X_test.to(device), y_test.to(device)
            test_preds = model(X_test).cpu().numpy()
            test_actuals = y_test.cpu().numpy()
            preds.append(test_preds)
            actuals.append(test_actuals)
    preds = np.concatenate(preds, axis=0)
    actuals = np.concatenate(actuals, axis=0)
    # Inverse transform to original scale
    preds_orig = scaler_y.inverse_transform(preds.reshape(-1, 1)).flatten()
    actuals_orig = scaler_y.inverse_transform(actuals.reshape(-1, 1)).flatten()
    mae  = mean_absolute_error(actuals_orig, preds_orig)
    rmse = np.sqrt(mean_squared_error(actuals_orig, preds_orig))
    mape = np.mean(np.abs((actuals_orig - preds_orig) /
                   (actuals_orig + 1e-8))) * 100
    actual_dir = np.sign(np.diff(actuals_orig))
    pred_dir   = np.sign(np.diff(preds_orig))
    dir_acc    = np.mean(actual_dir == pred_dir) * 100
    print(f"{model_name} Results")
    print(f"MAE             : {mae:.4f}")
    print(f"RMSE            : {rmse:.4f}")
    print(f"MAPE            : {mape:.2f}%")
    print(f"Directional Acc : {dir_acc:.2f}%")
    

In [ ]:
evaluate_model(trained_mlp, test_loader, train_dataset.scaler_y, model_name="MLP Forecaster")

MLP Forecaster Results
MAE             : 0.0891
RMSE            : 0.1425
MAPE            : 50.50%
Directional Acc : 54.10%


### LSTM Forecaster

In [ ]:
class LSTMForecaster(nn.Module):
    """
    Stacked LSTM forecaster.
    Input:  (batch, seq_len, n_features)
    Output: (batch,) when horizon=1, else (batch, horizon)
    """
    def __init__(self, input_size=1, hidden_size=64,
                 num_layers=2, horizon=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,          # (batch, seq, features)
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use final hidden state from last layer
        out = self.dropout(h_n[-1])   # (batch, hidden_size)
        return self.fc(out).squeeze(-1)  # (batch, horizon) -> (batch,) when horizon=1


In [ ]:
lstm_model = LSTMForecaster(input_size=data.shape[1]-1, hidden_size=64, num_layers=2)

In [ ]:
trained_lstm = train_model(lstm_model, train_loader, val_loader, epochs=20, lr=1e-3, model_name="LSTMForecaster")

Epoch 1/20  Train: 0.0256  Val: 0.0149  LR: 1.00e-03
Epoch 2/20  Train: 0.0173  Val: 0.0021  LR: 1.00e-03
Epoch 3/20  Train: 0.0067  Val: 0.0040  LR: 1.00e-03
Epoch 4/20  Train: 0.0036  Val: 0.0011  LR: 1.00e-03
Epoch 5/20  Train: 0.0027  Val: 0.0019  LR: 1.00e-03
Epoch 6/20  Train: 0.0021  Val: 0.0018  LR: 1.00e-03
Epoch 7/20  Train: 0.0020  Val: 0.0008  LR: 1.00e-03
Epoch 8/20  Train: 0.0022  Val: 0.0012  LR: 1.00e-03
Epoch 9/20  Train: 0.0017  Val: 0.0011  LR: 1.00e-03
Epoch 10/20  Train: 0.0014  Val: 0.0006  LR: 1.00e-03
Epoch 11/20  Train: 0.0015  Val: 0.0005  LR: 1.00e-03
Epoch 12/20  Train: 0.0012  Val: 0.0005  LR: 1.00e-03
Epoch 13/20  Train: 0.0012  Val: 0.0005  LR: 1.00e-03
Epoch 14/20  Train: 0.0013  Val: 0.0005  LR: 1.00e-03
Epoch 15/20  Train: 0.0012  Val: 0.0008  LR: 1.00e-03
Epoch 16/20  Train: 0.0013  Val: 0.0010  LR: 1.00e-03
Epoch 17/20  Train: 0.0011  Val: 0.0007  LR: 1.00e-03
Epoch 18/20  Train: 0.0012  Val: 0.0010  LR: 1.00e-03
Epoch 19/20  Train: 0.0012  Val: 0.00

In [ ]:
evaluate_model(trained_lstm, test_loader, train_dataset.scaler_y, model_name="LSTM Forecaster")

LSTM Forecaster Results
MAE             : 0.0343
RMSE            : 0.0496
MAPE            : 20.03%
Directional Acc : 60.66%


In [ ]:
### LSTM Hyperparameter Search
#Grid search over learning rates, epoch counts, and optimizers to find the best configuration.

In [ ]:
import itertools

def lstm_hparam_search(train_loader, val_loader, input_size,
                       lr_list, epoch_list, optimizer_list):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for lr, epochs, opt_name in itertools.product(lr_list, epoch_list, optimizer_list):
        model = LSTMForecaster(input_size=input_size, hidden_size=64, num_layers=2)
        model.to(device)

        if opt_name == "adamw":
            optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        elif opt_name == "rmsprop":
            optimizer = optim.RMSprop(model.parameters(), lr=lr)
        else:
            optimizer = optim.Adam(model.parameters(), lr=lr)

        criterion = nn.MSELoss()
        best_val = float("inf")

        for _ in range(epochs):
            model.train()
            for X_b, y_b in train_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                optimizer.zero_grad()
                loss = criterion(model(X_b), y_b.squeeze())
                loss.backward()
                optimizer.step()

            model.eval()
            val_losses = []
            with torch.no_grad():
                for X_v, y_v in val_loader:
                    X_v, y_v = X_v.to(device), y_v.to(device)
                    val_losses.append(criterion(model(X_v), y_v.squeeze()).item())
            best_val = min(best_val, np.mean(val_losses))

        results.append({"lr": lr, "epochs": epochs, "optimizer": opt_name, "best_val_loss": best_val})
        print(f"  lr={lr:.0e}  epochs={epochs:3d}  opt={opt_name:8s}  best_val_loss={best_val:.6f}")

    results_df = (pd.DataFrame(results)
                  .sort_values("best_val_loss")
                  .reset_index(drop=True))
    print("\nTop 5 configs:")
    print(results_df.head())
    return results_df

search_results = lstm_hparam_search(
    train_loader, val_loader,
    input_size=data.shape[1] - 1,
    lr_list=[1e-2, 1e-3, 5e-4],
    epoch_list=[20, 40],
    optimizer_list=["adam", "adamw", "rmsprop"],
)

  lr=1e-02  epochs= 20  opt=adam      best_val_loss=0.000579
  lr=1e-02  epochs= 20  opt=adamw     best_val_loss=0.000531
  lr=1e-02  epochs= 20  opt=rmsprop   best_val_loss=0.002022
  lr=1e-02  epochs= 40  opt=adam      best_val_loss=0.000489
  lr=1e-02  epochs= 40  opt=adamw     best_val_loss=0.000476
  lr=1e-02  epochs= 40  opt=rmsprop   best_val_loss=0.000825
  lr=1e-03  epochs= 20  opt=adam      best_val_loss=0.000507
  lr=1e-03  epochs= 20  opt=adamw     best_val_loss=0.000488
  lr=1e-03  epochs= 20  opt=rmsprop   best_val_loss=0.000526
  lr=1e-03  epochs= 40  opt=adam      best_val_loss=0.000448
  lr=1e-03  epochs= 40  opt=adamw     best_val_loss=0.000514
  lr=1e-03  epochs= 40  opt=rmsprop   best_val_loss=0.000467
  lr=5e-04  epochs= 20  opt=adam      best_val_loss=0.000531
  lr=5e-04  epochs= 20  opt=adamw     best_val_loss=0.000477
  lr=5e-04  epochs= 20  opt=rmsprop   best_val_loss=0.000472
  lr=5e-04  epochs= 40  opt=adam      best_val_loss=0.000446
  lr=5e-04  epochs= 40  

### LSTM V2 — Best Config with Optimizer & LR Scheduler
Uses the best (lr, epochs, optimizer) found above, plus `ReduceLROnPlateau` and gradient clipping.

In [ ]:
# train_model_v2 merged into train_model above.
# Equivalent call: train_model(..., optimizer_name="adamw", use_scheduler=True, clip_grad=1.0)

In [ ]:
best = search_results.iloc[0]
print(f"Best config — lr={best.lr:.0e}  epochs={int(best.epochs)}  optimizer={best.optimizer}  val_loss={best.best_val_loss:.6f}")

lstm_v2 = LSTMForecaster(input_size=data.shape[1] - 1, hidden_size=128, num_layers=2)
trained_lstm_v2 = train_model(
    lstm_v2, train_loader, val_loader,
    epochs=int(best.epochs),
    lr=best.lr,
    optimizer_name=best.optimizer,
    use_scheduler=True,
    clip_grad=1.0,
    model_name="LSTMForecaster_v2",
)

Best config — lr=5e-04  epochs=40  optimizer=adam  val_loss=0.000446
Epoch 1/40  Train: 0.0289  Val: 0.0164  LR: 5.00e-04
Epoch 2/40  Train: 0.0177  Val: 0.0032  LR: 5.00e-04
Epoch 3/40  Train: 0.0105  Val: 0.0017  LR: 5.00e-04
Epoch 4/40  Train: 0.0029  Val: 0.0015  LR: 5.00e-04
Epoch 5/40  Train: 0.0022  Val: 0.0011  LR: 5.00e-04
Epoch 6/40  Train: 0.0019  Val: 0.0016  LR: 5.00e-04
Epoch 7/40  Train: 0.0018  Val: 0.0008  LR: 5.00e-04
Epoch 8/40  Train: 0.0015  Val: 0.0009  LR: 5.00e-04
Epoch 9/40  Train: 0.0015  Val: 0.0006  LR: 5.00e-04
Epoch 10/40  Train: 0.0013  Val: 0.0005  LR: 5.00e-04
Epoch 11/40  Train: 0.0014  Val: 0.0008  LR: 5.00e-04
Epoch 12/40  Train: 0.0013  Val: 0.0005  LR: 5.00e-04
Epoch 13/40  Train: 0.0013  Val: 0.0005  LR: 5.00e-04
Epoch 14/40  Train: 0.0010  Val: 0.0005  LR: 5.00e-04
Epoch 15/40  Train: 0.0011  Val: 0.0007  LR: 5.00e-04
Epoch 16/40  Train: 0.0012  Val: 0.0018  LR: 5.00e-04
Epoch 17/40  Train: 0.0011  Val: 0.0008  LR: 5.00e-04
Epoch 18/40  Train: 0.

In [ ]:
evaluate_model(trained_lstm_v2, test_loader, train_dataset.scaler_y, model_name="LSTM V2 (Tuned)")

LSTM V2 (Tuned) Results
MAE             : 0.0318
RMSE            : 0.0478
MAPE            : 20.03%
Directional Acc : 58.20%


### LSTM V3 — HuberLoss + Larger Hidden + Higher Dropout
`hidden_size=256`, `num_layers=3`, `dropout=0.3`, `criterion=HuberLoss`

In [ ]:
lstm_v3 = LSTMForecaster(input_size=data.shape[1] - 1, hidden_size=256, num_layers=3, dropout=0.3)
trained_lstm_v3 = train_model(
    lstm_v3, train_loader, val_loader,
    epochs=int(best.epochs),
    lr=best.lr,
    optimizer_name=best.optimizer,
    criterion_name="huber",
    use_scheduler=True,
    clip_grad=1.0,
    model_name="LSTMForecaster_v3",
)

Epoch 1/40  Train: 0.0109  Val: 0.0013  LR: 5.00e-04
Epoch 2/40  Train: 0.0072  Val: 0.0010  LR: 5.00e-04
Epoch 3/40  Train: 0.0021  Val: 0.0006  LR: 5.00e-04
Epoch 4/40  Train: 0.0014  Val: 0.0007  LR: 5.00e-04
Epoch 5/40  Train: 0.0009  Val: 0.0005  LR: 5.00e-04
Epoch 6/40  Train: 0.0008  Val: 0.0005  LR: 5.00e-04
Epoch 7/40  Train: 0.0007  Val: 0.0004  LR: 5.00e-04
Epoch 8/40  Train: 0.0010  Val: 0.0007  LR: 5.00e-04
Epoch 9/40  Train: 0.0008  Val: 0.0004  LR: 5.00e-04
Epoch 10/40  Train: 0.0007  Val: 0.0003  LR: 5.00e-04
Epoch 11/40  Train: 0.0006  Val: 0.0008  LR: 5.00e-04
Epoch 12/40  Train: 0.0006  Val: 0.0005  LR: 5.00e-04
Epoch 13/40  Train: 0.0006  Val: 0.0003  LR: 5.00e-04
Epoch 14/40  Train: 0.0007  Val: 0.0004  LR: 5.00e-04
Epoch 15/40  Train: 0.0006  Val: 0.0005  LR: 5.00e-04
Epoch 16/40  Train: 0.0006  Val: 0.0003  LR: 2.50e-04
Epoch 17/40  Train: 0.0005  Val: 0.0003  LR: 2.50e-04
Epoch 18/40  Train: 0.0006  Val: 0.0004  LR: 2.50e-04
Epoch 19/40  Train: 0.0005  Val: 0.00

In [ ]:
evaluate_model(trained_lstm_v3, test_loader, train_dataset.scaler_y, model_name="LSTM V3 (Huber + Deeper)")

LSTM V3 (Huber + Deeper) Results
MAE             : 0.0394
RMSE            : 0.0532
MAPE            : 25.14%
Directional Acc : 55.74%


### CNN-LSTM Forecaster
using news around option 